In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col, upper, current_timestamp, avg, count

# -------------------------
# Define Schema
# -------------------------
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("price", IntegerType(), True)
])

# =========================
# 1️⃣ Bronze Layer
# =========================
bronze_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .schema(schema) \
    .load("/Volumes/workspace/default/airbnb_volume/")

bronze_query = bronze_stream.writeStream \
    .format("delta") \
    .option("mergeSchema", "true") \
    .option("checkpointLocation", "/Volumes/workspace/default/checkpoints/bronze_airbnb") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .table("workspace.default.bronze_airbnb")

bronze_query.awaitTermination()

# =========================
# 2️⃣ Silver Layer
# =========================
silver_stream = spark.readStream.table("workspace.default.bronze_airbnb")

silver_transformed = silver_stream \
    .filter(col("id").isNotNull()) \
    .withColumn("city", upper(col("city"))) \
    .withColumn("price", col("price").cast("int")) \
    .withColumn("ingestion_time", current_timestamp())

silver_query = silver_transformed.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/workspace/default/checkpoints/silver_airbnb") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .table("workspace.default.silver_airbnb")

silver_query.awaitTermination()

# =========================
# 3️⃣ Gold Layer
# =========================
gold_stream = spark.readStream.table("workspace.default.silver_airbnb")

gold_agg = gold_stream \
    .groupBy("city") \
    .agg(
        avg("price").alias("avg_price"),
        count("*").alias("total_listings")
    )

gold_query = gold_agg.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/workspace/default/checkpoints/gold_airbnb") \
    .outputMode("complete") \
    .trigger(availableNow=True) \
    .table("workspace.default.gold_airbnb")

gold_query.awaitTermination()